# EMEP country-level analysis: PM10, POD1 and POD3

This notebook reads the EMEP five-year-average NetCDF file and the grid-to-country fraction NetCDF file, calculates **area-weighted country means**, and exports an Excel workbook.

**Important choices to review:**
- `SURF_ug_PM10` is used for PM10 surface concentration (µg/m³), not the alternative `SURF_ug_PM10_rh50` field.
- EMEP contains several POD1 and POD3 variants. The main three-output table defaults to `POD1_DF` and `POD3_TC`. Change these below to the vegetation/impact indicators appropriate for your analysis.
- Country fractions are used as percentages (0–100), multiplied by grid-cell area. This correctly handles grid cells that straddle borders.
- The source is already a five-year average of 2016–2020. Results represent that average; they are not five separate annual observations.

In [1]:
# If needed, uncomment and run once to install NetCDF readers:
#%pip install xarray netCDF4 h5netcdf pandas openpyxl numpy

from pathlib import Path
import re
import numpy as np
import pandas as pd
import xarray as xr
import h5py, h5netcdf

print("xarray:", xr.__version__)


xarray: 2026.9.0


In [ ]:
DATA_FILE = Path(r"Raw_Data\EMEP_raw_data\v7aa\EMEP01_annual_baseline2040_5year_average.nc")

In [4]:
# Put both .nc files in the same folder as this notebook, or update these paths.
COUNTRY_FILE = Path(r"Raw_Data\EMEP_raw_data\emep_ll_gridfraction_01degCEIP_2018_ospar2020_eez.nc")
OUTPUT_XLSX = Path("Processed_Data/EMEP_country_PM10_POD1_POD3.xlsx")
OUTPUT_CSV = Path("Processed_Data/EMEP_country_PM10_POD1_POD3.csv")

for path in (DATA_FILE, COUNTRY_FILE):
    if not path.exists():
        raise FileNotFoundError(f"Cannot find {path.resolve()}. Update the path or move the file next to this notebook.")


emep = xr.open_dataset(DATA_FILE, engine="h5netcdf")
grid = xr.open_dataset(COUNTRY_FILE, engine="h5netcdf")

print("EMEP dimensions:", dict(emep.sizes))
print("Country grid dimensions:", dict(grid.sizes))
print("EMEP variables of interest:", [v for v in emep.data_vars if "PM10" in v or v.startswith("POD1") or v.startswith("POD3")])

EMEP dimensions: {'time': 1, 'lat': 520, 'lon': 1200}
Country grid dimensions: {'lat': 520, 'lon': 1200}
EMEP variables of interest: ['POD1_DF', 'POD1_CF', 'POD1_IAM_DF', 'POD1_IAM_DF_MED', 'POD1_SNL', 'POD1_SNL_MED', 'POD3_TC', 'POD3_IAM_WH', 'POD3_IAM_WH_Irrig', 'POD3_IAM_WH_MED', 'POD3_IAM_WH_MED_Irrig', 'SURF_ug_PM10', 'SURF_ug_PM10_rh50']


## Inspect available indicators

The source file has multiple POD variants. Review this list before finalising the indicator choices below.

In [5]:
for var in [v for v in emep.data_vars if v.startswith("POD1") or v.startswith("POD3") or "PM10" in v]:
    da = emep[var]
    print(f"{var:24s} dims={da.dims}, units={da.attrs.get('units', 'not specified')}, long_name={da.attrs.get('long_name', '')}")


POD1_DF                  dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_DF
POD1_CF                  dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_CF
POD1_IAM_DF              dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_IAM_DF
POD1_IAM_DF_MED          dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_IAM_DF_MED
POD1_SNL                 dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_SNL
POD1_SNL_MED             dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD1_SNL_MED
POD3_TC                  dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD3_TC
POD3_IAM_WH              dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD3_IAM_WH
POD3_IAM_WH_Irrig        dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD3_IAM_WH_Irrig
POD3_IAM_WH_MED          dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=POD3_IAM_WH_MED
POD3_IAM_WH_MED_Irrig    dims=('time', 'lat', 'lon'), units=mmole/m2, long_name=

In [6]:
# Select the variables for the main three-output dataframe.
PM10_VAR = "SURF_ug_PM10"  # µg/m³; alternative: SURF_ug_PM10_rh50
POD1_VAR = "POD1_DF"        # e.g. POD1_CF, POD1_SNL, POD1_IAM_DF
POD3_VAR = "POD3_TC"        # e.g. POD3_IAM_WH, POD3_IAM_WH_MED, etc.

for var in (PM10_VAR, POD1_VAR, POD3_VAR):
    if var not in emep:
        raise KeyError(f"Variable {var!r} not found. Choose from: {list(emep.data_vars)}")


## Identify country fraction layers

The grid-to-country file contains country layers named like `CC_6` with a two-letter `long_name` (for example `DK`) and percentage units. This excludes sea regions, OSPAR regions, EEZ layers and aggregate layers. The selection can be inspected before calculating.

In [29]:
mapping_countries = {
    "AL": "Albania",
    "AM": "Armenia",
    "AT": "Austria",
    "AZ": "Azerbaijan",
    "BA": "Bosnia-H",
    "BE": "Belgium",
    "BG": "Bulgaria",
    "BY": "Belarus",
    "CH": "Switzerland",
    "CY": "Cyprus",
    "CZ": "Czech Rep.",
    "DE": "Germany",
    "DK": "Denmark",
    "EE": "Estonia",
    "ES": "Spain",
    "FI": "Finland",
    "FR": "France",
    "GB": "United Kingdom",
    "GE": "Georgia",
    "GL": "Greenland",
    "GR": "Greece",
    "HR": "Croatia",
    "HU": "Hungary",
    "IE": "Ireland",
    "IS": "Iceland",
    "IT": "Italy",
    "KG": "Kyrgyzstan",
    "KZ": "Kazakhstan",
    "LI": "Liechtenstein",
    "LT": "Lithuania",
    "LU": "Luxembourg",
    "LV": "Latvia",
    "MC": "Monaco",
    "MD": "R Moldova",
    "ME": "Montenegro",
    "MK": "North Macedonia",
    "MT": "Malta",
    "NL": "Netherlands",
    "NO": "Norway",
    "PL": "Poland",
    "PT": "Portugal",
    "RO": "Romania",
    "RS": "Serbia",
    "RU": "Russia",
    "SE": "Sweden",
    "SI": "Slovenia",
    "SK": "Slovakia",
    "TJ": "Tajikistan",
    "TM": "Turkmenistan",
    "TR": "Türkiye",
    "UA": "Ukraine",
    "UZ": "Uzbekistan"
}

In [31]:
def get_attr_text(value):
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="replace")
    if isinstance(value, np.ndarray) and value.size == 1:
        return get_attr_text(value.item())
    return str(value) if value is not None else ""

country_layers = []
for name, da in grid.data_vars.items():
    units = get_attr_text(da.attrs.get("units", "")).strip()
    code = get_attr_text(da.attrs.get("long_name", "")).strip().upper()
    # The country layers in this file are percentage grids with two-character country/territory codes.
    if units == "%" and re.fullmatch(r"[A-Z]{2}", code) and da.ndim == 2:
        country_layers.append((name, code))

country_layers = sorted(country_layers, key=lambda x: x[1])
if not country_layers:
    raise ValueError("No country fraction layers found. Inspect grid.data_vars and adjust country-layer selection.")

country_inventory = pd.DataFrame(country_layers, columns=["grid_variable", "country_code"])
country_inventory["country_name"] = country_inventory["country_code"].replace(mapping_countries)

display(country_inventory)

,grid_variable,country_code,country_name
0,CC_1,AL,Albania
1,CC_56,AM,Armenia
2,CC_2,AT,Austria
3,CC_69,AZ,Azerbaijan
4,CC_50,BA,Bosnia-H
5,CC_3,BE,Belgium
6,CC_4,BG,Bulgaria
7,CC_39,BY,Belarus
8,CC_24,CH,Switzerland
9,CC_55,CY,Cyprus


## Calculate area-weighted country averages

For each country, the weight for each grid cell is `cell area × country fraction`. The fraction is converted from percent to a 0–1 share. Concentration and POD indicators are spatially averaged, not summed. This produces a representative country mean for each field.

In [40]:
# NEW code which produces values 

# ---- 1. Coordinate checks (then work with plain numpy, no xarray alignment) ----
for coord in ("lat", "lon"):
    if coord not in emep.coords or coord not in grid.coords:
        raise KeyError(f"Expected coordinate {coord!r} in both files.")
    if emep.sizes[coord] != grid.sizes[coord] or not np.allclose(emep[coord].values, grid[coord].values):
        raise ValueError(f"The {coord} coordinates do not match between the EMEP and country grid files.")

def to_2d(da, name, source="data"):
    """Return a float64 numpy array with shape (lat, lon); fill values -> NaN."""
    for dim in [d for d in da.dims if d not in ("lat", "lon")]:
        if da.sizes[dim] != 1:
            raise ValueError(f"{name} has {da.sizes[dim]} entries along {dim!r}; expected 1.")
        da = da.isel({dim: 0})
    arr = da.transpose("lat", "lon").values.astype("float64")

    # xarray usually moves _FillValue/missing_value into .encoding after decoding, so check both
    for key in ("_FillValue", "missing_value"):
        fill = da.attrs.get(key, da.encoding.get(key))
        if fill is not None:
            arr[arr == float(np.ravel(fill)[0])] = np.nan

    arr[~np.isfinite(arr) | (np.abs(arr) >= 1e20)] = np.nan
    return arr

# ---- 2. Cell area (km²; only used as a weight, so the unit cancels) ----
if "cell_area" not in grid:
    raise KeyError("The country grid file does not contain cell_area.")
cell_area = to_2d(grid["cell_area"], "cell_area")
cell_area = np.where(np.isfinite(cell_area) & (cell_area > 0), cell_area, 0.0)

# ---- 3. Fields to aggregate ----
fields = {
    "PM10_surface_ug_m3": to_2d(emep[PM10_VAR], PM10_VAR),
    "POD1": to_2d(emep[POD1_VAR], POD1_VAR),
    "POD3": to_2d(emep[POD3_VAR], POD3_VAR),
}

# Quick sanity check: if a field is all NaN here, the problem is the variable/masking, not the weighting
for name, arr in fields.items():
    n_valid = np.isfinite(arr).sum()
    print(f"{name:22s} valid cells: {n_valid:>8d} / {arr.size}  "
          f"min={np.nanmin(arr) if n_valid else np.nan:.4g}  max={np.nanmax(arr) if n_valid else np.nan:.4g}")

# ---- 4. Area-weighted country means ----
def country_weighted_mean(field, weights):
    """Weighted mean over cells where the field is valid; also returns data coverage (0-1)."""
    valid = np.isfinite(field)
    total_w = weights.sum()
    w = np.where(valid, weights, 0.0)
    w_sum = w.sum()
    if w_sum <= 0:
        return np.nan, 0.0
    mean = np.sum(np.where(valid, field, 0.0) * w) / w_sum
    coverage = w_sum / total_w if total_w > 0 else 0.0
    return mean, coverage

rows = []
for layer_name, country_code in country_layers:
    fraction = to_2d(grid[layer_name], layer_name)
    fraction = np.where(np.isfinite(fraction) & (fraction > 0), fraction, 0.0)
    weights = cell_area * fraction          # scale of 'fraction' (0-1 or 0-100) cancels in the mean

    row = {"country_code": country_code, "country_area_weight": weights.sum()}
    for output_name, field in fields.items():
        mean, coverage = country_weighted_mean(field, weights)
        row[output_name] = mean
        row[f"{output_name}_coverage"] = coverage
    rows.append(row)

country_df = pd.DataFrame(rows).sort_values("country_code").reset_index(drop=True)
country_df["POD1_variable"] = POD1_VAR
country_df["POD3_variable"] = POD3_VAR
country_df["country_name"] = country_df["country_code"].replace(mapping_countries)

country_df = country_df[['country_code', 'country_name', 'PM10_surface_ug_m3', 'POD1', 'POD3', 'POD1_variable', 'POD3_variable']]

PM10_surface_ug_m3     valid cells:   624000 / 624000  min=0.8699  max=412
POD1                   valid cells:   624000 / 624000  min=0  max=52.91
POD3                   valid cells:   624000 / 624000  min=0  max=14.66


## Optional: calculate every POD1 and POD3 variant

This extra table helps avoid losing information when the exact vegetation indicator has not yet been decided. The main output remains three selected indicators.

In [47]:
pod_vars = [v for v in emep.data_vars if v.startswith(("POD1", "POD3"))]

# Load each POD field once as a 2-D numpy array
pod_fields = {v: to_2d(emep[v], v) for v in pod_vars}

all_pod_rows = []
for layer_name, country_code in country_layers:
    fraction = to_2d(grid[layer_name], layer_name)
    fraction = np.where(np.isfinite(fraction) & (fraction > 0), fraction, 0.0)
    weights = cell_area * fraction

    row = {"country_code": country_code}
    for var, field in pod_fields.items():
        row[var], _ = country_weighted_mean(field, weights)   # drop the coverage value
    all_pod_rows.append(row)

all_pod_df = pd.DataFrame(all_pod_rows).sort_values("country_code").reset_index(drop=True)
all_pod_df.head()

,country_code,POD1_DF,POD1_CF,POD1_IAM_DF,POD1_IAM_DF_MED,POD1_SNL,POD1_SNL_MED,POD3_TC,POD3_IAM_WH,POD3_IAM_WH_Irrig,POD3_IAM_WH_MED,POD3_IAM_WH_MED_Irrig
0,AL,29.922404,19.965663,29.639406,52.463638,9.445587,12.222157,3.376892,12.651431,12.767385,10.623075,10.760569
1,AM,15.372506,6.310040,27.788740,0.000000,7.216312,0.000000,1.133047,5.456285,5.762391,0.000000,0.000000
2,AT,20.115296,19.183795,21.173321,3.203732,6.286525,0.549435,1.890969,8.179893,9.743481,0.419179,0.448323
3,AZ,10.749611,4.983855,22.346991,0.000000,8.018086,0.000000,4.075952,12.956035,15.547069,0.000000,0.000000
4,BA,28.287181,20.633681,26.768799,47.273218,8.402449,10.194642,2.642876,10.490686,10.790363,8.498829,8.809604


## Export results

The Excel workbook contains:
1. `Three_outputs` — one row per country with PM10, selected POD1 and selected POD3.
2. `All_POD_variants` — country means for all POD1/POD3 variables found in the source.
3. `Country_layers` — mapping between country codes and source grid variables.
4. `Metadata` — selected variables, units and aggregation method.

In [42]:
metadata_df = pd.DataFrame({
    "item": ["Source data file", "Country grid file", "Source period", "PM10 variable", "PM10 units", "POD1 variable", "POD1 units", "POD3 variable", "POD3 units", "Aggregation"],
    "value": [
        DATA_FILE.name,
        COUNTRY_FILE.name,
        "Five-year average 2016–2020 (not separate annual values)",
        PM10_VAR,
        get_attr_text(emep[PM10_VAR].attrs.get("units", "")),
        POD1_VAR,
        get_attr_text(emep[POD1_VAR].attrs.get("units", "")),
        POD3_VAR,
        get_attr_text(emep[POD3_VAR].attrs.get("units", "")),
        "Area-weighted country mean using grid-cell area × country fraction; missing cells excluded",
    ]
})


In [43]:
with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    country_df.to_excel(writer, sheet_name="Three_outputs", index=False)
    all_pod_df.to_excel(writer, sheet_name="All_POD_variants", index=False)
    country_inventory.to_excel(writer, sheet_name="Country_layers", index=False)
    metadata_df.to_excel(writer, sheet_name="Metadata", index=False)

country_df.to_csv(OUTPUT_CSV, index=False)

In [44]:
# Close datasets when finished.
emep.close()
grid.close()

## OLD Code

In [45]:
# OLD code which only resulted in NaN values in the output file 

# # Align data to the same latitude/longitude grid and remove the singleton time dimension.
# # The two files should have matching lat/lon coordinates; stop if they do not.
# for coord in ("lat", "lon"):
#     if coord not in emep.coords or coord not in grid.coords:
#         raise KeyError(f"Expected coordinate {coord!r} in both files.")
#     if emep.sizes.get(coord) != grid.sizes.get(coord) or not np.allclose(emep[coord].values, grid[coord].values):
#         raise ValueError(f"The {coord} coordinates do not match between the EMEP and country grid files.")

# # cell_area is in km². It is only used for weighting, so its unit cancels in the mean.
# if "cell_area" not in grid:
#     raise KeyError("The country grid file does not contain cell_area.")
# cell_area = grid["cell_area"].astype("float64")

# # Return a 2-D lat/lon field; expected source shape is (time, lat, lon) with one time record.
# def as_2d(varname):
#     da = emep[varname]
#     if "time" in da.dims:
#         if da.sizes["time"] != 1:
#             raise ValueError(f"{varname} has {da.sizes['time']} time records; this notebook expects one five-year-mean record.")
#         da = da.isel(time=0)
#     # Convert fill values and extreme sentinels to missing values.
#     da = da.astype("float64")
#     fill = da.attrs.get("_FillValue", None)
#     if fill is not None:
#         da = da.where(da != float(fill))
#     da = da.where(np.isfinite(da) & (np.abs(da) < 1e20))
#     return da

# fields = {
#     "PM10_surface_ug_m3": as_2d(PM10_VAR),
#     "POD1": as_2d(POD1_VAR),
#     "POD3": as_2d(POD3_VAR),
# }

# # Compute area-weighted means; missing data are excluded from both numerator and denominator.
# def country_weighted_mean(field, fraction_da):
#     fraction = fraction_da.astype("float64")
#     # Country grid fraction values are percentages; clip invalid values for safety.
#     fraction = fraction.where((fraction >= 0) & (fraction <= 100)) / 100.0
#     weights = (cell_area * fraction).where(np.isfinite(field) & np.isfinite(cell_area))
#     numerator = (field * weights).sum(dim=("lat", "lon"), skipna=True)
#     denominator = weights.where(np.isfinite(field)).sum(dim=("lat", "lon"), skipna=True)
#     return float((numerator / denominator).values) if float(denominator.values) > 0 else np.nan

# rows = []
# for layer_name, country_code in country_layers:
#     fraction = grid[layer_name]
#     row = {"country_code": country_code}
#     for output_name, field in fields.items():
#         row[output_name] = country_weighted_mean(field, fraction)
#     rows.append(row)

# country_df = pd.DataFrame(rows).sort_values("country_code").reset_index(drop=True)
# country_df["POD1_variable"] = POD1_VAR
# country_df["POD3_variable"] = POD3_VAR
# country_df


In [ ]:
# OLD version, doesn't work with new code fix

# pod_vars = [v for v in emep.data_vars if v.startswith("POD1") or v.startswith("POD3")]
# all_pod_rows = []
# for layer_name, country_code in country_layers:
#     row = {"country_code": country_code}
#     fraction = grid[layer_name]
#     for var in pod_vars:
#         row[var] = country_weighted_mean(as_2d(var), fraction)
#     all_pod_rows.append(row)
# all_pod_df = pd.DataFrame(all_pod_rows).sort_values("country_code").reset_index(drop=True)
# all_pod_df.head()